# Recurrent Neural Networks (RNNs)

Understanding sequence modeling with RNNs, LSTMs, and GRUs.

## Learning Objectives

- Understand recurrent architectures
- Learn about vanishing gradients
- Implement LSTM and GRU networks
- Process sequential data
- Build sequence-to-sequence models

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')
torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 1. RNN Fundamentals

RNNs process sequences by maintaining a hidden state that gets updated at each timestep.

In [ ]:
# Simple RNN cell from scratch
class SimpleRNNCell:
    """Manual RNN cell implementation."""
    
    def __init__(self, input_size, hidden_size):
        self.hidden_size = hidden_size
        
        # Initialize weights
        self.W_ih = np.random.randn(hidden_size, input_size) * 0.1  # input to hidden
        self.W_hh = np.random.randn(hidden_size, hidden_size) * 0.1  # hidden to hidden
        self.b_h = np.zeros((hidden_size, 1))
    
    def forward(self, x, h_prev):
        """Forward pass for one timestep."""
        # h_t = tanh(W_ih * x + W_hh * h_prev + b)
        h_new = np.tanh(self.W_ih @ x + self.W_hh @ h_prev + self.b_h)
        return h_new
    
    def process_sequence(self, X):
        """Process entire sequence."""
        seq_len = X.shape[1]
        h = np.zeros((self.hidden_size, 1))
        hidden_states = [h]
        
        for t in range(seq_len):
            x_t = X[:, t:t+1]
            h = self.forward(x_t, h)
            hidden_states.append(h)
        
        return hidden_states

# Demo
rnn_cell = SimpleRNNCell(input_size=3, hidden_size=4)
X = np.random.randn(3, 5)  # 3 features, 5 timesteps
states = rnn_cell.process_sequence(X)

print(f"Input shape: {X.shape}")
print(f"Number of hidden states: {len(states)}")
print(f"Final hidden state shape: {states[-1].shape}")

In [ ]:
# PyTorch RNN
print("=== PyTorch RNN ===")

# Input: (batch, seq_len, input_size)
batch_size = 2
seq_len = 10
input_size = 8
hidden_size = 16

rnn = nn.RNN(input_size=input_size, hidden_size=hidden_size, 
             num_layers=1, batch_first=True)

x = torch.randn(batch_size, seq_len, input_size)
h0 = torch.zeros(1, batch_size, hidden_size)  # Initial hidden state

output, h_n = rnn(x, h0)

print(f"Input shape: {x.shape}")
print(f"Output shape: {output.shape}  # All hidden states")
print(f"Final hidden state: {h_n.shape}")

## 2. LSTM (Long Short-Term Memory)

LSTMs solve the vanishing gradient problem with gating mechanisms.

In [ ]:
# LSTM gates visualization
fig, axes = plt.subplots(1, 4, figsize=(16, 3))

gates = ['Forget Gate', 'Input Gate', 'Cell Candidate', 'Output Gate']
descriptions = [
    'What to forget\nfrom cell state',
    'What new info\nto store',
    'New candidate\nvalues',
    'What to output\nfrom cell'
]
colors = ['#ff6b6b', '#4ecdc4', '#45b7d1', '#96ceb4']

for ax, gate, desc, color in zip(axes, gates, descriptions, colors):
    ax.add_patch(plt.Rectangle((0.2, 0.2), 0.6, 0.6, color=color, ec='black'))
    ax.text(0.5, 0.5, gate, ha='center', va='center', fontsize=12, fontweight='bold')
    ax.text(0.5, 0.05, desc, ha='center', va='center', fontsize=10)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis('off')

plt.suptitle('LSTM Gate Components', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# PyTorch LSTM
print("=== PyTorch LSTM ===")

lstm = nn.LSTM(input_size=input_size, hidden_size=hidden_size, 
               num_layers=2, batch_first=True, dropout=0.1)

x = torch.randn(batch_size, seq_len, input_size)
h0 = torch.zeros(2, batch_size, hidden_size)  # 2 layers
c0 = torch.zeros(2, batch_size, hidden_size)  # Cell state

output, (h_n, c_n) = lstm(x, (h0, c0))

print(f"Input shape: {x.shape}")
print(f"Output shape: {output.shape}")
print(f"Hidden state: {h_n.shape}")
print(f"Cell state: {c_n.shape}")
print(f"\nLSTM parameters: {sum(p.numel() for p in lstm.parameters()):,}")

## 3. GRU (Gated Recurrent Unit)

GRUs are a simpler variant of LSTMs with fewer parameters.

In [ ]:
# PyTorch GRU
print("=== PyTorch GRU ===")

gru = nn.GRU(input_size=input_size, hidden_size=hidden_size, 
             num_layers=2, batch_first=True)

x = torch.randn(batch_size, seq_len, input_size)
h0 = torch.zeros(2, batch_size, hidden_size)

output, h_n = gru(x, h0)

print(f"Output shape: {output.shape}")
print(f"Hidden state: {h_n.shape}")
print(f"\nGRU parameters: {sum(p.numel() for p in gru.parameters()):,}")
print(f"LSTM parameters: {sum(p.numel() for p in lstm.parameters()):,}")
print(f"\nGRU has ~75% of LSTM parameters")

## 4. Sequence Prediction Example

In [ ]:
# Generate sine wave data
def create_sine_data(n_samples=1000, seq_length=50):
    """Create sequences from sine wave for prediction."""
    t = np.linspace(0, 100, n_samples + seq_length)
    data = np.sin(t) + 0.1 * np.random.randn(len(t))
    
    X, y = [], []
    for i in range(n_samples):
        X.append(data[i:i+seq_length])
        y.append(data[i+seq_length])
    
    X = np.array(X).reshape(-1, seq_length, 1)
    y = np.array(y).reshape(-1, 1)
    
    return torch.FloatTensor(X), torch.FloatTensor(y)

X, y = create_sine_data()
print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")

# Split data
train_size = int(0.8 * len(X))
X_train, X_test = X[:train_size], X[train_size:]
y_train, y_test = y[:train_size], y[train_size:]

In [ ]:
# Visualize data
plt.figure(figsize=(14, 4))
plt.plot(range(100), X[0].flatten(), label='Input sequence')
plt.scatter([100], y[0], color='red', s=100, label='Target', zorder=5)
plt.xlabel('Timestep')
plt.ylabel('Value')
plt.title('Sequence Prediction Task')
plt.legend()
plt.show()

In [ ]:
class LSTMPredictor(nn.Module):
    """LSTM for time series prediction."""
    
    def __init__(self, input_size=1, hidden_size=32, num_layers=2):
        super(LSTMPredictor, self).__init__()
        
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, 
                            batch_first=True, dropout=0.2)
        self.fc = nn.Linear(hidden_size, 1)
    
    def forward(self, x):
        # LSTM output
        out, _ = self.lstm(x)
        
        # Use last timestep
        out = self.fc(out[:, -1, :])
        return out

model = LSTMPredictor().to(device)
print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# Training
from torch.utils.data import TensorDataset, DataLoader

train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

losses = []
epochs = 50

for epoch in range(epochs):
    model.train()
    epoch_loss = 0
    
    for batch_x, batch_y in train_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        
        optimizer.zero_grad()
        output = model(batch_x)
        loss = criterion(output, batch_y)
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
    
    avg_loss = epoch_loss / len(train_loader)
    losses.append(avg_loss)
    
    if (epoch + 1) % 10 == 0:
        print(f'Epoch [{epoch+1}/{epochs}], Loss: {avg_loss:.6f}')

In [ ]:
# Plot loss
plt.figure(figsize=(10, 4))
plt.plot(losses)
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.title('Training Loss')
plt.show()

In [ ]:
# Evaluate
model.eval()
with torch.no_grad():
    X_test_dev = X_test.to(device)
    predictions = model(X_test_dev).cpu().numpy()

# Plot predictions
plt.figure(figsize=(14, 5))
plt.plot(y_test.numpy()[:100], label='Actual', linewidth=2)
plt.plot(predictions[:100], label='Predicted', linewidth=2, linestyle='--')
plt.xlabel('Sample')
plt.ylabel('Value')
plt.title('LSTM Predictions vs Actual')
plt.legend()
plt.show()

# Metrics
mse = np.mean((predictions - y_test.numpy())**2)
print(f"Test MSE: {mse:.6f}")

## 5. Bidirectional RNNs

In [ ]:
# Bidirectional LSTM
bilstm = nn.LSTM(input_size=8, hidden_size=16, num_layers=1, 
                 batch_first=True, bidirectional=True)

x = torch.randn(2, 10, 8)
output, (h_n, c_n) = bilstm(x)

print("Bidirectional LSTM:")
print(f"  Input: {x.shape}")
print(f"  Output: {output.shape}  # hidden_size * 2 = 32")
print(f"  Hidden: {h_n.shape}  # 2 directions")

## 6. Text Classification Example

In [ ]:
# Simple text classifier
class TextClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_size, num_classes):
        super(TextClassifier, self).__init__()
        
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_size, batch_first=True, 
                            bidirectional=True)
        self.fc = nn.Linear(hidden_size * 2, num_classes)
        self.dropout = nn.Dropout(0.5)
    
    def forward(self, x):
        # Embed
        embedded = self.embedding(x)
        
        # LSTM
        out, (h_n, _) = self.lstm(embedded)
        
        # Concatenate final forward and backward hidden states
        hidden = torch.cat((h_n[-2], h_n[-1]), dim=1)
        
        # Classify
        out = self.dropout(hidden)
        out = self.fc(out)
        return out

# Demo
classifier = TextClassifier(vocab_size=10000, embed_dim=100, 
                           hidden_size=128, num_classes=2)
print(classifier)

# Test with dummy data
dummy_input = torch.randint(0, 10000, (4, 50))  # 4 sentences, max 50 words
output = classifier(dummy_input)
print(f"\nOutput shape: {output.shape}")

## 7. RNN Comparison

In [ ]:
import pandas as pd

comparison = pd.DataFrame({
    'Type': ['RNN', 'LSTM', 'GRU'],
    'Gates': ['None', 'Forget, Input, Output', 'Reset, Update'],
    'Cell State': ['No', 'Yes', 'No'],
    'Parameters': ['Fewest', 'Most', 'Medium'],
    'Long Dependencies': ['Poor', 'Good', 'Good'],
    'Training Speed': ['Fastest', 'Slowest', 'Medium']
})

print(comparison.to_string(index=False))

## 8. Key Takeaways

1. **RNNs** process sequential data with shared weights across timesteps
2. **Vanishing gradients** make vanilla RNNs struggle with long sequences
3. **LSTMs** use gating mechanisms to control information flow
4. **GRUs** are simpler alternatives to LSTMs with similar performance
5. **Bidirectional** RNNs capture context from both directions
6. **Embeddings** convert discrete tokens to dense vectors

### Common Applications
- Time series prediction
- Natural language processing
- Speech recognition
- Machine translation
- Sentiment analysis